<a href="https://colab.research.google.com/github/Ayushman125/Essentials_of_AI/blob/main/EAI_Lab4(b).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Experiment 4(b):

####Q) Write a python script to measure the proximity measure for the given dataset (Ordinal, Nominal, Binary(Asymmetric), Numerical) comprising mixed-type attributes.print the final dissimilarity matrix

### Proximity Formula for Mixed-Type Attributes (Gower's Distance)

The overall dissimilarity between object $i$ and object $j$ is defined as:

$$d(i,j) = \frac{\sum_{f=1}^{p} \delta_{ij}^{(f)} \cdot d_{ij}^{(f)}}{\sum_{f=1}^{p} \delta_{ij}^{(f)}}$$

Where:
* $d_{ij}^{(f)}$ is the dissimilarity contribution of attribute $f$ between objects $i$ and $j$.
* $\delta_{ij}^{(f)}$ is the indicator variable: $0$ if attribute $f$ is missing or if both values are $0$ in an asymmetric binary feature, and $1$ otherwise.

---

### Dissimilarity Formulas by Attribute Type

**1. Nominal Attribute**
$$d_{ij}^{(f)} = \begin{cases} 0, & \text{if } x_{if} = x_{jf} \\ 1, & \text{if } x_{if} \neq x_{jf} \end{cases} \quad \text{and} \quad \delta_{ij}^{(f)} = 1$$

**2. Asymmetric Binary Attribute**
$$d_{ij}^{(f)} = \begin{cases} 0, & \text{if } x_{if} = x_{jf} \\ 1, & \text{if } x_{if} \neq x_{jf} \end{cases}$$

$$\delta_{ij}^{(f)} = \begin{cases} 0, & \text{if } x_{if} = 0 \text{ and } x_{jf} = 0 \\ 1, & \text{otherwise} \end{cases}$$

**3. Ordinal Attribute**
First, map rank $r_{if} \in \{1, \dots, M_f\}$ to the range $[0, 1]$:
$$z_{if} = \frac{r_{if} - 1}{M_f - 1}$$

Then compute normalized difference:
$$d_{ij}^{(f)} = |z_{if} - z_{jf}| \quad \text{and} \quad \delta_{ij}^{(f)} = 1$$

**4. Numerical Attribute**
Normalized by the overall range of attribute $f$:
$$d_{ij}^{(f)} = \frac{|x_{if} - x_{jf}|}{\max_k(x_{kf}) - \min_k(x_{kf})} \quad \text{and} \quad \delta_{ij}^{(f)} = 1$$

In [ ]:
import numpy as np
import pandas as pd


def generate_mixed_dataset(num_records=5, random_state=42):
    """Generates a dataset with mixed attribute types:

    Nominal, Asymmetric Binary, Ordinal, and Numerical.
    """
    np.random.seed(random_state)

    data = {
        # Nominal: Categorical with no order
        "Nominal_Color": np.random.choice(
            ["Red", "Blue", "Green"], size=num_records
        ),
        # Asymmetric Binary: 1 is rare/important, 0 is absent
        # (e.g., test result)
        "AsymBinary_Test": [1, 0, 1, 0, 1],
        # Ordinal: Categorical with natural ordering
        "Ordinal_Size": np.random.choice(
            ["Small", "Medium", "Large"], size=num_records
        ),
        # Numerical: Continuous numeric value
        "Numerical_Age": np.random.randint(20, 60, size=num_records),
    }

    df = pd.DataFrame(data)
    return df


def compute_gower_dissimilarity(df, ordinal_ranks):
    """Computes the Gower Dissimilarity Matrix for a DataFrame with mixed-type

    attributes.
    """
    n = len(df)
    p = df.shape[1]
    dissimilarity_matrix = np.zeros((n, n))

    # Pre-process Ordinal attributes: Map to ranks and scale to [0, 1]
    df_processed = df.copy()
    for col, ranks in ordinal_ranks.items():
        # Map categories to integer ranks
        mapped_ranks = df[col].map(ranks)
        min_r = min(ranks.values())
        max_r = max(ranks.values())
        # Scale to range [0, 1] using z_if = (r_if - 1) / (M_f - 1)
        df_processed[col] = (mapped_ranks - min_r) / (max_r - min_r)

    # Compute pairwise dissimilarities
    for i in range(n):
        for j in range(n):
            if i == j:
                dissimilarity_matrix[i, j] = 0.0
                continue

            delta_sum = 0.0
            weighted_d_sum = 0.0

            for col in df.columns:
                val_i = df_processed.loc[i, col]
                val_j = df_processed.loc[j, col]

                # 1. Nominal Attribute
                if col.startswith("Nominal"):
                    delta = 1.0  # Indicator: 1 if valid comparison
                    d_f = 0.0 if val_i == val_j else 1.0

                # 2. Asymmetric Binary Attribute
                elif col.startswith("AsymBinary"):
                    # Indicator delta is 0 if both are 0 (asymmetric condition)
                    if val_i == 0 and val_j == 0:
                        delta = 0.0
                        d_f = 0.0
                    else:
                        delta = 1.0
                        d_f = 0.0 if val_i == val_j else 1.0

                # 3. Ordinal Attribute (treated as scaled numerical after rank mapping)
                elif col.startswith("Ordinal"):
                    delta = 1.0
                    d_f = abs(val_i - val_j)

                # 4. Numerical Attribute
                elif col.startswith("Numerical"):
                    delta = 1.0
                    col_range = df[col].max() - df[col].min()
                    d_f = (
                        abs(val_i - val_j) / col_range
                        if col_range != 0
                        else 0.0
                    )

                delta_sum += delta
                weighted_d_sum += delta * d_f

            # Calculate final Gower dissimilarity d(i, j)
            if delta_sum > 0:
                dissimilarity_matrix[i, j] = weighted_d_sum / delta_sum
            else:
                dissimilarity_matrix[i, j] = 0.0

    return pd.DataFrame(
        dissimilarity_matrix,
        index=[f"Object_{i}" for i in range(n)],
        columns=[f"Object_{i}" for i in range(n)],
    )


# Execution
if __name__ == "__main__":
    # 1. Generate Dataset
    df = generate_mixed_dataset(num_records=5)

    print("--- Generated Mixed-Attribute Dataset ---")
    print(df)
    print("\n" + "=" * 50 + "\n")

    # Define ranking order for Ordinal columns
    ordinal_ranks = {"Ordinal_Size": {"Small": 1, "Medium": 2, "Large": 3}}

    # 2. Compute Dissimilarity Matrix
    matrix = compute_gower_dissimilarity(df, ordinal_ranks)

    print("--- Dissimilarity Matrix d(i, j) ---")
    print(matrix.round(4))

--- Generated Mixed-Attribute Dataset ---
  Nominal_Color  AsymBinary_Test Ordinal_Size  Numerical_Age
0         Green                1        Small             30
1           Red                0        Large             30
2         Green                1       Medium             43
3         Green                0        Large             55
4           Red                1        Large             59


--- Dissimilarity Matrix d(i, j) ---
          Object_0  Object_1  Object_2  Object_3  Object_4
Object_0    0.0000    0.7500    0.2371    0.7155    0.7500
Object_1    0.7500    0.0000    0.7371    0.6207    0.5000
Object_2    0.2371    0.7371    0.0000    0.4784    0.5129
Object_3    0.7155    0.6207    0.4784    0.0000    0.5345
Object_4    0.7500    0.5000    0.5129    0.5345    0.0000
